#### Noisy Quantum SVM - Spambase

In [1]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 1.4.4
Aer: 0.17.2
QML: 0.8.4


In [2]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [3]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
# from imblearn.over_sampling import RandomOverSampler  # For optional balancing

In [4]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, PegasosQSVC

In [5]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make",
    "word_freq_address",
    "word_freq_all",
    "word_freq_3d",
    "word_freq_our",
    "word_freq_over",
    "word_freq_remove",
    "word_freq_internet",
    "word_freq_order",
    "word_freq_mail",
    "word_freq_receive",
    "word_freq_will",
    "word_freq_people",
    "word_freq_report",
    "word_freq_addresses",
    "word_freq_free",
    "word_freq_business",
    "word_freq_email",
    "word_freq_you",
    "word_freq_credit",
    "word_freq_your",
    "word_freq_font",
    "word_freq_000",
    "word_freq_money",
    "word_freq_hp",
    "word_freq_hpl",
    "word_freq_george",
    "word_freq_650",
    "word_freq_lab",
    "word_freq_labs",
    "word_freq_telnet",
    "word_freq_857",
    "word_freq_data",
    "word_freq_415",
    "word_freq_85",
    "word_freq_technology",
    "word_freq_1999",
    "word_freq_parts",
    "word_freq_pm",
    "word_freq_direct",
    "word_freq_cs",
    "word_freq_meeting",
    "word_freq_original",
    "word_freq_project",
    "word_freq_re",
    "word_freq_edu",
    "word_freq_table",
    "word_freq_conference",
    "char_freq_;",
    "char_freq_(",
    "char_freq_[",
    "char_freq_!",
    "char_freq_$",
    "char_freq_#",
    "capital_run_length_average",
    "capital_run_length_longest",
    "capital_run_length_total",
    # finally the target label column:
    "label"
]

# --- 1. Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

In [6]:
# 2. Some basic processing
print(f"Original shape of Spambase data: {df.shape}") # Prints original dataset shape
df.drop_duplicates(inplace=True) # Remove duplicates
print(f"Shape after dropping duplicates: {df.shape}\n") # Then print again the new shape

Original shape of Spambase data: (4210, 58)
Shape after dropping duplicates: (4210, 58)



In [7]:
# Data Preparation

# 1. Split features and target
X = df.drop('label', axis=1)
y = df['label']

# ============================================
# SUBSET DATA (for QSVC - 300 samples)
# ============================================
# First sample 429 samples from full dataset
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    train_size=429,
    stratify=y,
    random_state=42
)

# Then do 70:30 split on this subset
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset,
    test_size=0.30,
    random_state=42,
    stratify=y_subset
)
# This gives you ~300 training, ~129 test samples for QSVC

print(f"QSVC Training set: {X_train.shape[0]} samples")
print(f"QSVC Test set: {X_test.shape[0]} samples")

# ============================================
# FULL DATA (for PegasosQSVC - 4000+ samples)
# ============================================
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"PegasosQSVC Training set: {X_train_full.shape[0]} samples")
print(f"PegasosQSVC Test set: {X_test_full.shape[0]} samples\n")

QSVC Training set: 300 samples
QSVC Test set: 129 samples
PegasosQSVC Training set: 2947 samples
PegasosQSVC Test set: 1263 samples



In [8]:
# Scaling for SUBSET (QSVC)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaler

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

# Scaling for FULL DATASET (PegasosQSVC)
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

X_train_full_scaled_df = pd.DataFrame(X_train_full_scaled, columns=X.columns)
X_test_full_scaled_df = pd.DataFrame(X_test_full_scaled, columns=X.columns)

In [9]:
# --- Feature Correlation Analysis ---
print("--- Feature Correlation Analysis ---")
THRESH = 0.9

# Calculate correlation matrix on the SCALED TRAINING data (subset)
corr_matrix_train = X_train_scaled_df.corr().abs()

# Get the upper triangle of the correlation matrix
upper_triangle = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Find features with correlation greater than the threshold
columns_to_drop = set()
for column in upper_triangle.columns:
    high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
    if high_corr_partners:
        for partner in high_corr_partners:
            # IMPORTANT: Check correlation with the TRAINING target variable
            corr_main_vs_target = y_train.corr(X_train_scaled_df[column])
            corr_partner_vs_target = y_train.corr(X_train_scaled_df[partner])
            
            print(f"Found pair: ('{column}', '{partner}') with correlation > {THRESH}")
            if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                columns_to_drop.add(column)
                print(f"-> Dropping '{column}' (weaker correlation with target)")
            else:
                columns_to_drop.add(partner)
                print(f"-> Dropping '{partner}' (weaker correlation with target)")

to_drop_final = sorted(list(columns_to_drop))
print(f"\nTotal features to drop ({len(to_drop_final)}): {to_drop_final}")

# Drop the identified columns from SUBSET (QSVC)
X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)

# Drop the same columns from FULL DATASET (PegasosQSVC)
X_train_full_selected = X_train_full_scaled_df.drop(columns=to_drop_final)
X_test_full_selected = X_test_full_scaled_df.drop(columns=to_drop_final)

print(f"\nOriginal number of features: {X_train.shape[1]}")
print(f"Number of features after selection: {X_train_selected.shape[1]}\n")

--- Feature Correlation Analysis ---
Found pair: ('word_freq_415', 'word_freq_857') with correlation > 0.9
-> Dropping 'word_freq_857' (weaker correlation with target)

Total features to drop (1): ['word_freq_857']

Original number of features: 57
Number of features after selection: 56



c:\Users\User\anaconda3\envs\qsvm_conda\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\User\anaconda3\envs\qsvm_conda\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [10]:
# PCA for SUBSET (QSVC)
n_components = 4
pca = PCA(n_components=n_components, random_state=42)

# Fit on the selected training data and transform both sets
X_train_pca = pca.fit_transform(X_train_selected)
X_test_pca = pca.transform(X_test_selected)

print(f"QSVC - Shape after PCA (Train): {X_train_pca.shape}")
print(f"QSVC - Shape after PCA (Test):  {X_test_pca.shape}")

# PCA for FULL DATASET (PegasosQSVC)
pca_full = PCA(n_components=n_components, random_state=42)
X_train_full_pca = pca_full.fit_transform(X_train_full_selected)
X_test_full_pca = pca_full.transform(X_test_full_selected)

print(f"PegasosQSVC - Shape after PCA (Train): {X_train_full_pca.shape}")
print(f"PegasosQSVC - Shape after PCA (Test):  {X_test_full_pca.shape}")

QSVC - Shape after PCA (Train): (300, 4)
QSVC - Shape after PCA (Test):  (129, 4)
PegasosQSVC - Shape after PCA (Train): (2947, 4)
PegasosQSVC - Shape after PCA (Test):  (1263, 4)


##### Noise Simulation Setup

In [11]:
# Noise Model implementation (Depolarizing error and Readout Error)
print("--- Setting up Noise Model ---")

p_gate_1q = 0.001   # 0.1% error for single-qubit gates (u1, u2, u3)
p_gate_2q = 0.01    # 1.0% error for two-qubit gates (cx)
p_readout = 0.02    # 2.0% chance of wrong measurement

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(p_gate_1q, 1), ['u1', 'u2', 'u3'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(p_gate_2q, 2), ['cx'])
readout_error = ReadoutError([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
noise_model.add_all_qubit_readout_error(readout_error)

print("Noise model created.")

--- Setting up Noise Model ---
Noise model created.


##### Backend, Sampler and Pass Manager Implementation

In [12]:
# Noisy backend
noisy_backend = AerSimulator(
    noise_model=noise_model,
    seed_simulator=12345,
)

# Noisy Sampler
noise_sampler = AerSampler.from_backend(
    backend=noisy_backend,
    default_shots=256, # Remember to change 
)

# Transpilation pass manager
pm = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)

print("Noisy backend, sampler and pass manager ready!")

Noisy backend, sampler and pass manager ready!


##### Quantum Kernel Implementation

In [13]:
# Feature map setup
feature_dim = n_components
fm = ZZFeatureMap(feature_dimension=feature_dim, reps=1, entanglement='linear')

# Fidelity with noisy sampler and transpilation
fidelity = ComputeUncompute(sampler=noise_sampler, pass_manager=pm)

# Noisy quantum kernel
noisy_qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm, max_circuits_per_job=1000)

print("Quantum kernel created.")

Quantum kernel created.


##### Compute Kernel Matrices

In [14]:
# Compute Kernel Matrices (once, upfront)
print("Computing kernel matrices...")
start_kernel = time.time()

# For QSVC subset
matrix_train = noisy_qkernel.evaluate(x_vec=X_train_pca)
matrix_test = noisy_qkernel.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

# For PegasosQSVC full dataset
matrix_train_full = noisy_qkernel.evaluate(x_vec=X_train_full_pca)
matrix_test_full = noisy_qkernel.evaluate(x_vec=X_test_full_pca, y_vec=X_train_full_pca)

kernel_time = time.time() - start_kernel
print(f"Kernel matrices computed in {kernel_time:.2f} seconds.")

Computing kernel matrices...
Kernel matrices computed in 32973.94 seconds.


##### Save Kernel Matrices

In [ ]:
import pickle
import numpy as np

# Save all kernel matrices to disk
np.save('matrix_train.npy', matrix_train)
np.save('matrix_test.npy', matrix_test)
np.save('matrix_train_full.npy', matrix_train_full)
np.save('matrix_test_full.npy', matrix_test_full)

print("✓ All kernel matrices saved to disk!")
print(f"  - matrix_train.npy ({matrix_train.shape})")
print(f"  - matrix_test.npy ({matrix_test.shape})")
print(f"  - matrix_train_full.npy ({matrix_train_full.shape})")
print(f"  - matrix_test_full.npy ({matrix_test_full.shape})")

In [15]:
# ===== LOAD PRE-COMPUTED KERNEL MATRICES FROM DISK =====
import numpy as np

print("Loading pre-computed kernel matrices from disk...")

matrix_train = np.load('matrix_train.npy')
matrix_test = np.load('matrix_test.npy')
matrix_train_full = np.load('matrix_train_full.npy')
matrix_test_full = np.load('matrix_test_full.npy')

print("✓ All kernel matrices loaded!")
print(f"  - matrix_train: {matrix_train.shape}")
print(f"  - matrix_test: {matrix_test.shape}")
print(f"  - matrix_train_full: {matrix_train_full.shape}")
print(f"  - matrix_test_full: {matrix_test_full.shape}")


Loading pre-computed kernel matrices from disk...
✓ All kernel matrices loaded!
  - matrix_train: (300, 300)
  - matrix_test: (129, 300)
  - matrix_train_full: (2947, 2947)
  - matrix_test_full: (1263, 2947)


##### QSVC Implementation

In [16]:
print("="*70)
print(">>> PHASE 1: NOISY QSVM with SVC (300 Train Samples)")
print("="*70)

# Hyperparameter tuning with precomputed kernel
param_grid = {'C': [0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    SVC(kernel='precomputed', class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=1
)

start_time_sub = time.time()
grid_search.fit(matrix_train, y_train)
qsvc_model = grid_search.best_estimator_
train_time_sub = time.time() - start_time_sub

print(f"Best parameters: {grid_search.best_params_}")
print()

# Predictions
y_train_pred = qsvc_model.predict(matrix_train)
y_test_pred = qsvc_model.predict(matrix_test)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
generalization_gap = abs(train_accuracy - test_accuracy)

# Print formatted output (matching your Lung Cancer style)
print(f"\n--- Noisy QSVM Evaluation (Spambase) ---")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:     {test_accuracy:.4f}")
print(f"Generalization Gap: {generalization_gap:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, zero_division=0))

print(f"Samples Used: {X_train_pca.shape[0]}")
print(f"Training Time: {train_time_sub:.2f} seconds\n")

>>> PHASE 1: NOISY QSVM with SVC (300 Train Samples)
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters: {'C': 100}


--- Noisy QSVM Evaluation (Spambase) ---
Training Accuracy: 1.0000
Test Accuracy:     0.5271
Generalization Gap: 0.4729

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.62      0.56      0.59        78
           1       0.41      0.47      0.44        51

    accuracy                           0.53       129
   macro avg       0.52      0.52      0.52       129
weighted avg       0.54      0.53      0.53       129

Samples Used: 300
Training Time: 0.10 seconds



##### PegasosQSVC

In [17]:
print("="*70)
print(f">>> PHASE 2: PEGASOS QSVC (Full {X_train_full_pca.shape[0]} Samples)")
print("="*70)

# Reset index for Pegasos
y_train_full_reset = y_train_full.reset_index(drop=True)

# Use the best C from SVC grid search
best_C = grid_search.best_params_['C']

# Use PRECOMPUTED kernel mode
pegasos_qsvc = PegasosQSVC(
    quantum_kernel=None,        
    C=best_C,
    num_steps=5000,
    precomputed=True            
)

start_time_peg = time.time()
pegasos_qsvc.fit(matrix_train_full, y_train_full_reset)  # ← Use matrix!
train_time_peg = time.time() - start_time_peg

# Predictions - use pre-computed matrices
y_train_pred_peg = pegasos_qsvc.predict(matrix_train_full)
y_test_pred_peg = pegasos_qsvc.predict(matrix_test_full)  # ← Use matrix!

train_accuracy_peg = accuracy_score(y_train_full, y_train_pred_peg)
test_accuracy_peg = accuracy_score(y_test_full, y_test_pred_peg)
generalization_gap_peg = abs(train_accuracy_peg - test_accuracy_peg)

# Print formatted output (matching your Lung Cancer style)
print(f"\n--- Noisy PegasosQSVC Evaluation (Spambase) ---")
print(f"Training Accuracy: {train_accuracy_peg:.4f}")
print(f"Test Accuracy:     {test_accuracy_peg:.4f}")
print(f"Generalization Gap: {generalization_gap_peg:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_full, y_test_pred_peg, zero_division=0))

print(f"\nSamples Used: {X_train_full_pca.shape[0]}")
print(f"Training Time: {train_time_peg:.2f} seconds")


>>> PHASE 2: PEGASOS QSVC (Full 2947 Samples)

--- Noisy PegasosQSVC Evaluation (Spambase) ---
Training Accuracy: 0.6013
Test Accuracy:     0.6010
Generalization Gap: 0.0003

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.60      1.00      0.75       759
           1       0.00      0.00      0.00       504

    accuracy                           0.60      1263
   macro avg       0.30      0.50      0.38      1263
weighted avg       0.36      0.60      0.45      1263


Samples Used: 2947
Training Time: 165.69 seconds


##### Model Evaluation

In [18]:
print("\n" + "="*70)
print("MODEL EVALUATION SUMMARY")
print("="*70)

print(f"{'Metric':<25} | {'QSVC (300 Samples)':<20} | {'PegasosQSVC (Full)':<20}")
print("-" * 70)
print(f"{'Samples Used':<25} | {X_train_pca.shape[0]:<20} | {X_train_full_pca.shape[0]:<20}")
print(f"{'Training Time (s)':<25} | {train_time_sub:<20.2f} | {train_time_peg:<20.2f}")
print(f"{'Training Accuracy':<25} | {train_accuracy:<20.4f} | {train_accuracy_peg:<20.4f}")
print(f"{'Test Accuracy':<25} | {test_accuracy:<20.4f} | {test_accuracy_peg:<20.4f}")
print(f"{'Generalization Gap':<25} | {generalization_gap:<20.4f} | {generalization_gap_peg:<20.4f}")
print("="*70)



MODEL EVALUATION SUMMARY
Metric                    | QSVC (300 Samples)   | PegasosQSVC (Full)  
----------------------------------------------------------------------
Samples Used              | 300                  | 2947                
Training Time (s)         | 0.10                 | 165.69              
Training Accuracy         | 1.0000               | 0.6013              
Test Accuracy             | 0.5271               | 0.6010              
Generalization Gap        | 0.4729               | 0.0003              


##### Overall Parameter


In [19]:
# Shots : 256 - can try with 8192
# Optimization Level : 1
# PCA : 4
# QSVC : Got hyperparameter from GridSearchCV - 300 samples
# Pegasos : Got hyperparameter from GridSearchCV from pervious QSVC calculation. - full dataset.